In [ ]:
!git clone --depth 1 https://github.com/langchain-ai/lca-lc-foundations.git
%cd lca-lc-foundations
!pwd
!pip install -r requirements.txt


## Initialising and invoking a model

In [ ]:
from google.colab import userdata


# Get the secret value from Colab's secrets manager
openai_key = userdata.get('OPENAI_API_KEY')



In [ ]:
!pip install langchain-openai
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-5-nano", api_key=openai_key)

In [ ]:
response = model.invoke("What's the capital of the Moon?")

In [ ]:
print(response.content)

In [ ]:
from pprint import pprint

pprint(response.response_metadata)

## Customising your Model

In [ ]:
model = init_chat_model(
    model="gpt-5-nano",api_key=openai_key,
    # Kwargs passed to the model:
    temperature=1.0
)

response = model.invoke("What's the capital of the Moon?")
print(response.content)

## Model Providers

https://docs.langchain.com/oss/python/integrations/chat

In [ ]:
model = init_chat_model(model="claude-sonnet-4-5")

response = model.invoke("What's the capital of the Moon?")
print(response.content)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

response = model.invoke("What's the capital of the Moon?")
print(response.content)

## Initialising and invoking an agent

In [ ]:
from langchain.agents import create_agent
system_prompt="""
You are a fictional sports writer, create a favour sport for the city the user requests. Please keep the structure below
Sport: The name of the sport played in the city
Number of players: Number of players who are required to play the sport
Past famous player: Name of the the iconic player of the city

"""

agent = create_agent(model=model,system_prompt=system_prompt)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the favourtie sport in Bengaluru?")]}
)

In [ ]:
from pprint import pprint

pprint(response)

In [ ]:
print(response['messages'][-1].content)

In [ ]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the favourtie sport in Bengaluru?"),
    AIMessage(content="'Sport: Bengaluru Circuitball\nNumber of players: 7 per side\nPast famous player: Anil Kumble'"),
    HumanMessage(content="Interesting, tell me more about Bengaluru Circuitball")]}
)

pprint(response)

## Streaming Output

In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="What's the favourtie sport in Bengaluru?"),
    AIMessage(content="'Sport: Bengaluru Circuitball\nNumber of players: 7 per side\nPast famous player: Anil Kumble'"),
    HumanMessage(content="Interesting, tell me more about Bengaluru Circuitball")]},
    stream_mode="messages"
):

    # token is a message chunk with token content
    # metadata contains which node produced the token

    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

# **###################  Tools  #############################**

In [ ]:
from langchain.agents import create_agent

model = init_chat_model(model="gpt-5-nano", api_key=openai_key)

no_tool_agent = create_agent(
    model=model
)

In [ ]:
# from pprint import pprint
# response=model.invoke("Is there a war between USA and Iran at the moment?")
# print(response.content)

from langchain.messages import HumanMessage

response = no_tool_agent.invoke(
    {"messages": [HumanMessage(content="Is there a war between USA and Iran at the moment?")]}
)
print(response['messages'][-1].content)


## Add Web Search Tool

In [ ]:
!pip install tavily-python
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from google.colab import userdata

# Get the Tavily API key from Colab's secrets manager
tavily_key = userdata.get('TAVILY_API_KEY')
tavily_client = TavilyClient(api_key=tavily_key)

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""
    return tavily_client.search(query)

tool_agent = create_agent(
    model=model,
    tools=[web_search]

)

response = tool_agent.invoke(
     {"messages": [HumanMessage(content="What is the model you are using?")]}
)

pprint(response)



#######    Memory   #######

In [ ]:
from langchain.agents import create_agent

model = init_chat_model(model="gpt-5-nano", api_key=openai_key)

no_memory_agent = create_agent(
    model=model
)

In [ ]:

response = no_memory_agent.invoke(
    {"messages": [HumanMessage(content="Hello my name is Vinay and my favourite colour is green")]}
)

print(response['messages'][-1].content)

question = HumanMessage(content="What is my favourite colour?")

response = no_memory_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
)

In [ ]:
question = HumanMessage(content="Hello my name is Vinay and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = memory_agent.invoke(
    {"messages": [question]},
    {"configurable": {"thread_id": "1"}},
)

print(response['messages'][-1].content)

question = HumanMessage(content="What is my name?")

response = memory_agent.invoke(
    {"messages": [question]} ,{"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)